# ViSQOL Evaluation — 10s segments (NO resample, LSB-safe)
**Fix:** Bỏ hoàn toàn `-ar 48000` trong ffmpeg. Giữ nguyên 44.1kHz để bảo toàn LSB watermark.
ViSQOL sẽ báo warning sample rate — chấp nhận, không ảnh hưởng kết quả MOS-LQO.

In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
import os

# Khôi phục visqol binary từ Drive
if not os.path.exists('/content/visqol/bazel-bin/visqol'):
    os.makedirs('/content/visqol/bazel-bin', exist_ok=True)
    os.makedirs('/content/visqol/model', exist_ok=True)
    !cp /content/drive/MyDrive/visqol_build/visqol /content/visqol/bazel-bin/
    !cp -r /content/drive/MyDrive/visqol_build/model/* /content/visqol/model/
    !chmod +x /content/visqol/bazel-bin/visqol
    print(' Đã khôi phục visqol binary từ Drive')
else:
    print(' visqol binary đã sẵn có')

 visqol binary đã sẵn có


In [42]:
import os, csv, random, subprocess
import numpy as np
import soundfile as sf

# ===== CONFIG =====
SEGMENT_DURATION_S = 10.0
N_SAMPLES          = 116
OUTPUT_BASE        = '/content/segments_10s'
RANDOM_SEED        = 42

cover_dir    = '/content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train'
ablation_base = '/content/drive/MyDrive/HK1-20252026/Steganography/Ablation'
VISQOL_BIN   = '/content/visqol/bazel-bin/visqol'
VISQOL_MODEL = '/content/visqol/model/libsvm_nu_svr_model.txt'
RESULTS_DIR  = '/content/drive/MyDrive/visqol_build'

random.seed(RANDOM_SEED)

# ===== HELPERS — KHÔNG resample =====
def get_segment_start(src_path, duration_s):
    return 0.0

def extract_only(src_path, dst_path, duration_s, start_s):
    """Cắt đoạn bằng soundfile Python — 100% bit-exact, không qua ffmpeg/codec."""
    info = sf.info(src_path)
    sr   = info.samplerate
    if info.duration <= start_s:
        start_s = 0.0
    start_frame = int(start_s * sr)
    n_frames    = int(min(duration_s, info.duration - start_s) * sr)
    data, sr_read = sf.read(src_path, start=start_frame, frames=n_frames, always_2d=False)
    sf.write(dst_path, data, sr_read, subtype=info.subtype)

def prepare_cover_segments(cover_dir, sample_files, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    starts = {}
    for i, fname in enumerate(sample_files):
        src = os.path.join(cover_dir, fname)
        dst = os.path.join(out_dir, fname)
        try:
            start_s = get_segment_start(src, SEGMENT_DURATION_S)
            extract_only(src, dst, SEGMENT_DURATION_S, start_s)
            starts[fname] = start_s
        except Exception as e:
            print(f'  [ERROR] {fname}: {e}')
        if (i + 1) % 25 == 0:
            print(f'  {i+1}/{len(sample_files)} done')
    print(f'✅ Cover segments: {out_dir} ({len(starts)} files)')
    return starts

def prepare_stego_segments(stego_dir, sample_files, cover_starts, out_dir, cfg_name):
    os.makedirs(out_dir, exist_ok=True)
    matched = 0
    for fname in sample_files:
        if fname not in cover_starts:
            continue
        src = os.path.join(stego_dir, fname)
        if not os.path.exists(src):
            continue
        dst = os.path.join(out_dir, fname)
        try:
            extract_only(src, dst, SEGMENT_DURATION_S, cover_starts[fname])
            matched += 1
        except Exception as e:
            print(f'  [ERROR] {fname}: {e}')
    print(f'✅ {cfg_name}: {matched}/{len(sample_files)} stego segments')

def build_batch_csv(cfg, stego_seg_dir, cover_seg_dir):
    cover_files = set(os.listdir(cover_seg_dir))
    stego_files = os.listdir(stego_seg_dir)
    pairs = [(os.path.join(cover_seg_dir, f), os.path.join(stego_seg_dir, f))
             for f in stego_files if f in cover_files]
    csv_path = f'batch_{cfg}_10s.csv'
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['reference', 'degraded'])
        w.writerows(pairs)
    print(f'  {csv_path}: {len(pairs)} cặp')
    return csv_path


In [43]:
# Xoá segments cũ (nếu có) để chạy sạch
import shutil
if os.path.exists(OUTPUT_BASE):
    shutil.rmtree(OUTPUT_BASE)
    print(f'Đã xoá {OUTPUT_BASE}')

# Chọn 100 file ngẫu nhiên (seed cố định)
all_cover_files = sorted(os.listdir(cover_dir))
sample_files = random.sample(all_cover_files, min(N_SAMPLES, len(all_cover_files)))
print(f'[*] Đã chọn {len(sample_files)} file (seed={RANDOM_SEED})')

# Cắt cover segments
cover_seg_dir = os.path.join(OUTPUT_BASE, 'cover')
cover_starts = prepare_cover_segments(cover_dir, sample_files, cover_seg_dir)

Đã xoá /content/segments_10s
[*] Đã chọn 116 file (seed=42)
  25/116 done
  50/116 done
  75/116 done
  100/116 done
✅ Cover segments: /content/segments_10s/cover (116 files)


In [44]:
#  cover vs stego M1 phải KHÁC NHAU
# Nếu MSE=0.0 → vẫn đang resample → dừng lại, kiểm tra lại
stego_verify_dir = f'{ablation_base}/1_NoRandom'
files_to_check = [f for f in sorted(os.listdir(cover_seg_dir))[:5]
                  if os.path.exists(os.path.join(stego_verify_dir, f))]

# Tạm cắt 1 file stego M1 để verify
os.makedirs('/tmp/verify_stego', exist_ok=True)
all_ok = True
for fname in files_to_check:
    src = os.path.join(stego_verify_dir, fname)
    dst = f'/tmp/verify_stego/{fname}'
    extract_only(src, dst, SEGMENT_DURATION_S, cover_starts.get(fname, 0.0))
    c, _ = sf.read(os.path.join(cover_seg_dir, fname))
    s, _ = sf.read(dst)
    L = min(len(c), len(s))
    mse = np.mean((c[:L].astype(np.float64) - s[:L].astype(np.float64))**2)
    identical = np.array_equal(c[:L], s[:L])
    status = 'ok' if not identical else '❌ IDENTICAL — vẫn bị resample!'
    print(f'{fname}: Identical={identical}, MSE={mse:.2e}  {status}')
    if identical:
        all_ok = False

if all_ok:
    print('\n VERIFY PASSED — LSB watermark được bảo toàn, tiếp tục chạy ViSQOL')
else:
    print('\n VERIFY FAILED — Dừng lại, kiểm tra lại hàm extract_only')

AM Contra - Heart Peripheral_drums.wav: Identical=False, MSE=4.32e-10  ok
Actions - Devil's Words_bass.wav: Identical=False, MSE=4.32e-10  ok
Actions - Devil's Words_mixture.wav: Identical=False, MSE=4.30e-10  ok
Actions - One Minute Smile_bass.wav: Identical=False, MSE=4.31e-10  ok
Actions - One Minute Smile_mixture.wav: Identical=False, MSE=4.32e-10  ok

 VERIFY PASSED — LSB watermark được bảo toàn, tiếp tục chạy ViSQOL


In [45]:
# ===== Cắt stego segments cho tất cả methods =====
# Phase Coding → force PCM_16 (ViSQOL chỉ đọc 16-bit)
import numpy as np

def extract_pcm16(src, dst, duration_s):
    """Bit-exact cut + force PCM_16."""
    info = sf.info(src)
    n = int(min(duration_s, info.duration) * info.samplerate)
    data, sr = sf.read(src, frames=n, always_2d=False)
    if data.dtype != np.int16:
        data = (data * 32767).clip(-32768, 32767).astype(np.int16)
    sf.write(dst, data, sr, subtype='PCM_16')

BAD_CHARS = set("&',()!")
BAD_FILES = {'Johnny Lokke - Whisper To A Scream_vocals.wav'}
def is_bad(f): return f in BAD_FILES or any(c in f for c in BAD_CHARS)

METHODS_FOLDERS = [
    ('M1',          '1_NoRandom'),
    ('M2',          '2_Random_Fixed_DefaultSalt'),
    ('M3',          '3_Random_Fixed_ContentSalt'),
    ('M4',          '4_Random_Adaptive_DefaultSalt'),
    ('M5',          '5_Random_Adaptive_ContentSalt'),
    ('PhaseCoding', '7_PhaseCoding'),
    ('Alarood2022', '8_Alarood2022_RandLSB'),
]

# Lấy sample_files sạch
clean_sample = [f for f in sample_files if not is_bad(f)]
print(f'Clean sample files: {len(clean_sample)}')

for cfg, folder in METHODS_FOLDERS:
    src_dir = os.path.join(ablation_base, folder)
    out_dir = os.path.join(OUTPUT_BASE, folder)
    os.makedirs(out_dir, exist_ok=True)
    cut, cached, err = 0, 0, 0
    for fname in clean_sample:
        dst = os.path.join(out_dir, fname)
        if os.path.exists(dst): cached += 1; continue
        src = os.path.join(src_dir, fname)
        if not os.path.exists(src): continue
        try:
            extract_pcm16(src, dst, SEGMENT_DURATION_S)
            cut += 1
        except Exception as e:
            print(f'  [ERR] {fname}: {e}'); err += 1
    print(f'✅ {cfg}: {cut} cut, {cached} cached, {err} err')

Clean sample files: 104
✅ M1: 104 cut, 0 cached, 0 err
✅ M2: 104 cut, 0 cached, 0 err
✅ M3: 104 cut, 0 cached, 0 err
✅ M4: 104 cut, 0 cached, 0 err
✅ M5: 104 cut, 0 cached, 0 err
✅ PhaseCoding: 100 cut, 0 cached, 0 err
✅ Alarood2022: 104 cut, 0 cached, 0 err


In [46]:
def run_visqol(csv_path, cfg):
    out_csv = f'{RESULTS_DIR}/results_{cfg}_10s.csv'
    os.system(f'{VISQOL_BIN} '
              f'--batch_input_csv {csv_path} '
              f'--results_csv {out_csv} '
              f'--similarity_to_quality_model {VISQOL_MODEL} '
              f'2>/dev/null')
    if os.path.exists(out_csv):
        import pandas as pd
        df = pd.read_csv(out_csv)
        print(f'Columns: {df.columns.tolist()}')  # xem tên cột thật
        # Tìm cột MOS
        mos_col = [c for c in df.columns if 'mos' in c.lower()]
        if mos_col:
            vals = pd.to_numeric(df[mos_col[0]], errors='coerce').dropna()
            print(f'✅ {cfg}: {len(vals)} files, MOS-LQO mean={vals.mean():.4f} ± {vals.std():.4f}')
        else:
            print(f'Columns không có MOS: {df.columns.tolist()}')
            print(df.head(3))
    else:
        print(f'❌ {cfg}: không có output CSV')

In [47]:
import csv, os, pandas as pd, subprocess, soundfile as sf, numpy as np

BAD_CHARS = set("&',()!")
BAD_FILES = {'Johnny Lokke - Whisper To A Scream_vocals.wav'}
VISQOL_BIN    = '/content/visqol/bazel-bin/visqol'
VISQOL_MODEL  = '/content/visqol/model/libsvm_nu_svr_model.txt'
RESULTS_DIR   = '/content/drive/MyDrive/visqol_build'
cover_seg_dir = '/content/segments_10s/cover'

# Lấy danh sách file SẠCH từ cover (loại bad)
clean_cover = sorted([f for f in os.listdir(cover_seg_dir)
                      if f not in BAD_FILES
                      and not any(c in f for c in BAD_CHARS)])
print(f'Clean cover files: {len(clean_cover)}')

METHODS = {
    'M1':          '/content/segments_10s/1_NoRandom',
    'M2':          '/content/segments_10s/2_Random_Fixed_DefaultSalt',
    'M3':          '/content/segments_10s/3_Random_Fixed_ContentSalt',
    'M4':          '/content/segments_10s/4_Random_Adaptive_DefaultSalt',
    'M5':          '/content/segments_10s/5_Random_Adaptive_ContentSalt',
    'PhaseCoding': '/content/segments_10s/7_PhaseCoding',
    'Alarood2022': '/content/segments_10s/8_Alarood2022_RandLSB',
}

def make_pairs(stego_dir):
    stego_set = set(os.listdir(stego_dir)) if os.path.exists(stego_dir) else set()
    return [(os.path.join(cover_seg_dir, f), os.path.join(stego_dir, f))
            for f in clean_cover if f in stego_set]

# Tìm N chung = số file có mặt trong TẤT CẢ methods
common = set(clean_cover)
for cfg, d in METHODS.items():
    if os.path.exists(d):
        present = set(os.listdir(d)) & set(clean_cover)
        common &= present
        print(f'{cfg}: {len(present)} files')

print(f'\nCommon (có trong tất cả): {len(common)} files')

Clean cover files: 104
M1: 104 files
M2: 104 files
M3: 104 files
M4: 104 files
M5: 104 files
PhaseCoding: 100 files
Alarood2022: 104 files

Common (có trong tất cả): 100 files


In [48]:

common_list = sorted(common)
print(f'Rerun M1, M5, PhaseCoding, Alarood2022 với {len(common_list)} files')

def run_one(cfg, stego_dir, chunk_size=40):
    pairs = [(os.path.join(cover_seg_dir, f), os.path.join(stego_dir, f))
             for f in common_list
             if os.path.exists(os.path.join(stego_dir, f))]
    print(f'{cfg}: {len(pairs)} pairs')

    csv_path = f'batch_{cfg}_common.csv'
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['reference','degraded'])
        w.writerows(pairs)

    all_dfs = []
    for i in range(0, len(pairs), chunk_size):
        chunk = pairs[i:i+chunk_size]
        tmp_in  = f'/tmp/c_{cfg}_{i}.csv'
        tmp_out = f'/tmp/c_{cfg}_{i}_out.csv'
        if os.path.exists(tmp_out): os.remove(tmp_out)
        with open(tmp_in, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['reference','degraded'])
            w.writerows(chunk)
        os.system(f'{VISQOL_BIN} '
                  f'--batch_input_csv {tmp_in} '
                  f'--results_csv {tmp_out} '
                  f'--similarity_to_quality_model {VISQOL_MODEL} '
                  f'2>/dev/null')
        if os.path.exists(tmp_out):
            all_dfs.append(pd.read_csv(tmp_out))
            print(f'  chunk {i//chunk_size+1} ✅')
        else:
            print(f'  chunk {i//chunk_size+1} FAILED')

    if not all_dfs:
        print(f'❌ {cfg}: failed'); return
    df_all = pd.concat(all_dfs, ignore_index=True)
    out = f'{RESULTS_DIR}/results_{cfg}_10s.csv'
    df_all.to_csv(out, index=False)
    mos = pd.to_numeric(df_all['moslqo'], errors='coerce').dropna()
    print(f'✅ {cfg}: N={len(mos)}, MOS-LQO={mos.mean():.4f} ± {mos.std():.4f}')

run_one('M1',          '/content/segments_10s/1_NoRandom')
run_one('M5',          '/content/segments_10s/5_Random_Adaptive_ContentSalt')
run_one('PhaseCoding', '/content/segments_10s/7_PhaseCoding')
run_one('Alarood2022', '/content/segments_10s/8_Alarood2022_RandLSB')

Rerun M1, M5, PhaseCoding, Alarood2022 với 100 files
M1: 100 pairs
  chunk 1 ✅
  chunk 2 ✅
  chunk 3 ✅
✅ M1: N=100, MOS-LQO=4.7295 ± 0.0085
M5: 100 pairs
  chunk 1 ✅
  chunk 2 ✅
  chunk 3 ✅
✅ M5: N=100, MOS-LQO=4.7302 ± 0.0078
PhaseCoding: 100 pairs
  chunk 1 ✅
  chunk 2 ✅
  chunk 3 ✅
✅ PhaseCoding: N=100, MOS-LQO=3.9135 ± 0.9165
Alarood2022: 100 pairs
  chunk 1 ✅
  chunk 2 ✅
  chunk 3 ✅
✅ Alarood2022: N=100, MOS-LQO=4.7302 ± 0.0078
